# 4) Congestion and nodal prices — Exercises

**Goals:** reproduce DC flows from the PTDFs, decompose nodal prices line by line, follow cost, rent and prices as lines tighten, value a line reinforcement with its shadow price, and explain nodal prices outside the range of every generator's cost.

The setup cell holds the lecture's network, plants and `PTDF`, the generator-to-bus map `G`, the lecture's `dispatch(limits)` — now accepting one limit for all lines or a dict `{line: MW}` (other lines unlimited) — and `pypsa_network(limits)`. Power in MW, prices in €/MWh, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

warnings.filterwarnings("ignore")
for name in ("pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

# ---- the lecture's network, plants and PTDFs ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM = 0.32
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")


def ptdf(reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(LINES), len(BUSES)))
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))                            # generator-to-bus map
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1


def as_limits(limits):
    """A number (every line) or a dict {line: MW} (others unlimited) -> Series of limits."""
    if isinstance(limits, (int, float)):
        return pd.Series(float(limits), index=LINES.index)
    return pd.Series({line: float(limits.get(line, 1e5)) for line in LINES.index})


def dispatch(limits):
    """The lecture's DC-constrained dispatch, with one limit for all lines or a limit per line."""
    limits = as_limits(limits)
    gi = pd.Index(GENS.index, name="gen")
    flow_per_gen, flow_of_load = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[gi]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray(flow_per_gen[l], coords=[gi])).sum()
        m.add_constraints(flow <= limits[line] + flow_of_load[l], name=f"upper {line}")
        m.add_constraints(flow >= -limits[line] + flow_of_load[l], name=f"lower {line}")
    m.add_objective((p * xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[gi])).sum())
    m.solve(solver_name="highs", output_flag=False)
    lam = float(m.constraints["balance"].dual)
    mu = pd.Series({line: float(m.constraints[f"upper {line}"].dual) + float(m.constraints[f"lower {line}"].dual)
                    for line in LINES.index})
    p_sol = p.solution.to_pandas()
    flows = pd.Series(PTDF.to_numpy() @ (G @ p_sol.to_numpy() - LOAD.to_numpy()), index=LINES.index)
    return {"cost": float(m.objective.value), "dispatch": p_sol, "flows": flows, "lmp": lam + PTDF.T @ mu,
            "lambda": lam, "mu": mu, "limits": limits}


def pypsa_network(limits):
    """The same network in PyPSA (call .optimize() or .lpf() on it)."""
    limits = as_limits(limits)
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    for bus, load in LOAD.items():
        if load:
            n.add("Load", f"{bus} load", bus=bus, p_set=load)
    for name, g in GENS.iterrows():
        n.add("Generator", name, bus=g.bus, p_nom=g.p_max_MW, marginal_cost=g.cost_EUR_per_MWh)
    for name, line in LINES.iterrows():
        n.add("Line", name, bus0=line.bus0, bus1=line.bus1, x=X_KM * line.km, r=0.0, s_nom=limits[name])
    return n

### Warm-ups

**1. Flows from the PTDFs**

Write `ptdf_flows(dispatch_mw)` returning the line flows (MW) caused by a dispatch (a Series over `GENS.index`) and the loads. Check it on the lecture's uncongested dispatch against PyPSA's DC power flow (`lpf()`) with the generators fixed at that dispatch.

In [ ]:
def ptdf_flows(dispatch_mw: pd.Series) -> pd.Series:
    """Line flows (MW) = PTDF x (generation - load) at each bus."""
    ...  # YOUR CODE HERE

In [ ]:
free = dispatch(2000)
n = pypsa_network(2000)
n.generators.p_set = free["dispatch"][n.generators.index]
n.lpf()
assert np.abs(ptdf_flows(free["dispatch"]).to_numpy() - n.lines_t.p0.iloc[0].to_numpy()).max() < 1e-6
ptdf_flows(free["dispatch"]).round(1)

**2. Where a nodal price comes from**

Write `decompose(res)` returning a table with one row per bus: λ, the contribution PTDF_ℓn · μ_ℓ of each **congested** line, and their sum. Check the sum against `res["lmp"]` and against PyPSA's nodal prices for the lecture's 400 MW case.

In [ ]:
def decompose(res: dict) -> pd.DataFrame:
    """lambda + per-line contributions PTDF[l, n] * mu[l] for the congested lines, and the total."""
    ...  # YOUR CODE HERE

In [ ]:
res400 = dispatch(400)
parts = decompose(res400)
n = pypsa_network(400)
n.optimize(solver_name="highs", log_to_console=False, progress=False)
assert np.abs(parts["LMP"].to_numpy() - res400["lmp"].to_numpy()).max() < 1e-9
assert np.abs(parts["LMP"].to_numpy() - n.buses_t.marginal_price.iloc[0][BUSES].to_numpy()).max() < 1e-6
parts.round(2)

### Core

**3. Tightening the lines**

For a common limit of [1000, 600, 500, 450, 400, 300] MW on every line, report: system cost, congestion rent, curtailed wind, number of congested lines, and the spread between the highest and lowest nodal price. Write `limit_sweep(limits)`.

In [ ]:
def limit_sweep(limits=(1000, 600, 500, 450, 400, 300)) -> pd.DataFrame:
    """Cost, rent, curtailment, congested lines and price spread for each common line limit."""
    ...  # YOUR CODE HERE

In [ ]:
sweep = limit_sweep()
assert sweep.cost_EUR_per_h.is_monotonic_increasing
assert sweep.loc[1000, "congested_lines"] == 0 and np.isclose(sweep.loc[1000, "rent_EUR_per_h"], 0, atol=1e-6)
sweep.round(1)

**4. What is a reinforcement worth?**

At the 400 MW limit, raise the limit of **one** congested line at a time — by 1 MW and by 100 MW, all other lines at 400 MW — and compare the cost saving with the line's shadow price: |μ| × 1 MW and |μ| × 100 MW. Write `reinforcement_value()`.

In [ ]:
def reinforcement_value() -> pd.DataFrame:
    """Cost saving of +1 MW and +100 MW on each congested line, against |mu| times the increase."""
    ...  # YOUR CODE HERE

In [ ]:
value = reinforcement_value()
assert np.allclose(value["saving_+1MW"], value["mu_x_1MW"], rtol=1e-6), "the shadow price is the value of the first MW"
value.round(1)

**5. Interpretation**

Why are the nodal prices identical at 400 and 300 MW although cost and dispatch change? What does the sweep show about congestion rent as lines tighten? How well does the shadow price predict the value of +1 MW and of +100 MW, and why does it fail for the larger step?

*Your answer here.*

### Challenge

**6. Prices no generator would offer**

Every generator here costs between 0 and 130 €/MWh. Run two cases with **only one** line limited (all others unlimited): (a) Zagreb–Rijeka at 200 MW, (b) Rijeka–Velebit at 250 MW. Report all nodal prices and decompose Rijeka's price; confirm both cases in PyPSA. Write `outside_the_range()`.

In [ ]:
def outside_the_range() -> dict:
    """Nodal prices and Rijeka's decomposition for the two single-line cases."""
    ...  # YOUR CODE HERE

In [ ]:
cases = outside_the_range()
prices = pd.DataFrame({k: v["lmp"] for k, v in cases.items()})
assert prices.loc["Rijeka"].min() < 0 and prices.loc["Rijeka"].max() > 130
print("Rijeka's price, decomposed:")
print(pd.DataFrame({k: v["rijeka"] for k, v in cases.items()}).round(2).fillna(""))
prices.round(2)

**7. Interpretation**

Explain in words why one more MW of load at Rijeka is worth −39 €/MWh in case (a) and costs 143 €/MWh in case (b). What would a generator at Rijeka be paid in case (a), and what does that say about where new generation should — or should not — be built?

*Your answer here.*